可预期业务错误 → 资源不存在、权限不足、状态冲突
未知服务端错误 → 代码 Bug、数据库断开、未处理异常

### HTTPException
FastAPI 使用 HTTPException 表达可预期的 HTTP 错误：

In [ ]:
from fastapi import FastAPI, HTTPException, status


app = FastAPI()


@app.get("/users/{user_id}")
async def get_user(user_id: int):
    user = None

    if user is None:
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND,
            detail="User not found",
        )

    return user

#### 为什么使用 raise？
HTTPException 是要抛出的异常，不是普通响应数据。

执行到 raise 后，当前处理函数立即停止。

## 自定义业务异常

In [ ]:
class AppError(Exception):
    def __init__(
        self,
        code: str,
        message: str,
        status_code: int,
        details=None,
    ):
        self.code = code
        self.message = message
        self.status_code = status_code
        self.details = details

# 注册全局处理器
from fastapi import Request
from fastapi.responses import JSONResponse
@app.exception_handler(AppError)
async def handle_app_error(
    request: Request,
    exc: AppError,
):
    return JSONResponse(
        status_code=exc.status_code,
        content={
            "error": {
                "code": exc.code,
                "message": exc.message,
                "details": exc.details,
            }
        },
    )

In [ ]:
# 业务端只负责抛出异常
@app.get("/users/{user_id}")
async def get_user(user_id: int):
    user = None

    if user is None:
        raise AppError(
            code="USER_NOT_FOUND",
            message="User not found",
            status_code=404,
        )

    return user

- 所有接口错误格式一致；
- 业务代码不需要重复构建 JSON；
- 前端可以根据稳定的 code 处理错误；
- 日志、请求 ID 和错误详情可以统一添加；
- HTTP 状态码与业务错误码分离。

In [ ]:
# 通用处理
import logging

from fastapi import Request
from fastapi.responses import JSONResponse


logger = logging.getLogger(__name__)


@app.exception_handler(Exception)
async def handle_unknown_error(
    request: Request,
    exc: Exception,
):
    logger.exception("Unhandled error")

    return JSONResponse(
        status_code=500,
        content={
            "error": {
                "code": "INTERNAL_SERVER_ERROR",
                "message": "Internal server error",
            }
        },
    )

不要到处`try{}except xxx`
问题包括：
- 容易误返回 200；
- 重复代码多；
- 可能吞掉 Bug；
- 错误格式不统一；
- traceback 容易丢失。

更合理的是：
- 业务层 → 抛出明确业务异常
- 全局异常处理器 → 转换为 HTTP 响应
- 未知异常处理器 → 记录日志并返回通用 500